# Warehouse Robot 
Task 1, Intelligent programming

## 1. Maps

In [3]:
MAP_A = [
    "S..#....",
    "##.#.##.",
    "....~~..",
    ".####.#.",
    "...~..#G"
]

MAP_B = [
    "S~~~~~~~G",
    ".#######.",
    ".........",
    ".#.#.#.#.",
    "........."
]

MAP_C = [
    "S..#....",
    ".#.#.##.",
    "...#..#.",
    ".###.##.",
    "....#..G"
]

print("MAP A:")
for row in MAP_A:
    print(row)

print("MAP B:")
for row in MAP_B:
    print(row)

print("MAP C:")
for row in MAP_C:
    print(row)

MAP A:
S..#....
##.#.##.
....~~..
.####.#.
...~..#G
MAP B:
S~~~~~~~G
.#######.
.........
.#.#.#.#.
.........
MAP C:
S..#....
.#.#.##.
...#..#.
.###.##.
....#..G


## 2. Helper functions

In [4]:
import heapq
import time

# look through the grid and return the (row, col) of S or G
def find_cell(grid, symbol):
    for r in range(len(grid)):
        for c in range(len(grid[r])):
            if grid[r][c] == symbol:
                return (r, c)
    return None


# cells the robot can move to from pos (up, down, left, right)
def neighbors(grid, pos):
    r, c = pos
    moves = [(r - 1, c), (r + 1, c), (r, c - 1), (r, c + 1)]
    result = []
    for nr, nc in moves:
        # has to be inside the map and not a wall
        if 0 <= nr < len(grid) and 0 <= nc < len(grid[0]):
            if grid[nr][nc] != "#":
                result.append((nr, nc))
    return result


# cost of stepping INTO a cell
def move_cost(grid, pos):
    r, c = pos
    if grid[r][c] == "~":
        return 3
    return 1


# manhattan distance, used as the heuristic for A*
def manhattan(pos, goal):
    return abs(pos[0] - goal[0]) + abs(pos[1] - goal[1])


# go back through parent to build the path from S to G
def build_path(parent, goal):
    path = []
    node = goal
    while node is not None:
        path.append(node)
        node = parent[node]
    path.reverse()
    return path


# print the map with * on the path
def draw_path(grid, path):
    drawing = [list(row) for row in grid]
    for r, c in path:
        if drawing[r][c] not in ("S", "G"):
            drawing[r][c] = "*"
    for row in drawing:
        print("".join(row))

## 3. UCS

In [5]:
def ucs(grid):
    start = find_cell(grid, "S")
    goal = find_cell(grid, "G")

    frontier = [(0, start)]      # (cost so far, position)
    best_cost = {start: 0}
    parent = {start: None}
    nodes_expanded = 0
    max_frontier = 1

    while frontier:
        cost, current = heapq.heappop(frontier)

        # skip old entries, we already found a cheaper way to this cell
        if cost != best_cost[current]:
            continue

        nodes_expanded += 1

        if current == goal:
            return {"path": build_path(parent, goal), "cost": cost,
                    "nodes_expanded": nodes_expanded, "max_frontier": max_frontier}

        for nxt in neighbors(grid, current):
            new_cost = cost + move_cost(grid, nxt)
            if nxt not in best_cost or new_cost < best_cost[nxt]:
                best_cost[nxt] = new_cost
                parent[nxt] = current
                heapq.heappush(frontier, (new_cost, nxt))

        max_frontier = max(max_frontier, len(frontier))

    # frontier is empty and we never reached G
    return {"path": None, "cost": None,
            "nodes_expanded": nodes_expanded, "max_frontier": max_frontier}

## 4. A*
Same as UCS but the queue is sorted by f = g + h, where h is the Manhattan distance to G.

In [6]:
def astar(grid):
    start = find_cell(grid, "S")
    goal = find_cell(grid, "G")

    frontier = [(manhattan(start, goal), 0, start)]   # (f, g, position)
    best_cost = {start: 0}
    parent = {start: None}
    nodes_expanded = 0
    max_frontier = 1

    while frontier:
        f, cost, current = heapq.heappop(frontier)

        if cost != best_cost[current]:
            continue

        nodes_expanded += 1

        if current == goal:
            return {"path": build_path(parent, goal), "cost": cost,
                    "nodes_expanded": nodes_expanded, "max_frontier": max_frontier}

        for nxt in neighbors(grid, current):
            new_cost = cost + move_cost(grid, nxt)
            if nxt not in best_cost or new_cost < best_cost[nxt]:
                best_cost[nxt] = new_cost
                parent[nxt] = current
                new_f = new_cost + manhattan(nxt, goal)
                heapq.heappush(frontier, (new_f, new_cost, nxt))

        max_frontier = max(max_frontier, len(frontier))

    return {"path": None, "cost": None,
            "nodes_expanded": nodes_expanded, "max_frontier": max_frontier}

## 5. Running the algorithms and printing the results

In [7]:
# runs one algorithm and times it
def run_and_measure(algorithm, grid):
    t = time.perf_counter()
    result = algorithm(grid)
    ms = (time.perf_counter() - t) * 1000
    path = result["path"]
    return {
        "moves": len(path) - 1 if path else None,
        "cost": result["cost"],
        "expanded": result["nodes_expanded"],
        "frontier": result["max_frontier"],
        "time_ms": ms,
        "path": path
    }


# prints UCS and A* for one map
def show(grid):
    for name, algorithm in (("UCS", ucs), ("A*", astar)):
        s = run_and_measure(algorithm, grid)
        print(name)
        if s["path"] is None:
            print("No path found")
        else:
            print("Path:", s["path"])
            print("Moves:", s["moves"], "| Cost:", s["cost"])
            draw_path(grid, s["path"])
        print("Nodes expanded:", s["expanded"], "| Max frontier:", s["frontier"],
              "| Time (ms):", round(s["time_ms"], 4))
        print()

## 6. Choose the map
Change `GRID` to `MAP_B` or `MAP_C` to test the other maps.

In [8]:
GRID = MAP_A
show(GRID)

UCS
Path: [(0, 0), (0, 1), (0, 2), (1, 2), (2, 2), (2, 3), (2, 4), (2, 5), (2, 6), (2, 7), (3, 7), (4, 7)]
Moves: 11 | Cost: 15
S**#....
##*#.##.
..******
.####.#*
...~..#G
Nodes expanded: 28 | Max frontier: 4 | Time (ms): 0.2833

A*
Path: [(0, 0), (0, 1), (0, 2), (1, 2), (2, 2), (2, 3), (2, 4), (2, 5), (2, 6), (2, 7), (3, 7), (4, 7)]
Moves: 11 | Cost: 15
S**#....
##*#.##.
..******
.####.#*
...~..#G
Nodes expanded: 21 | Max frontier: 5 | Time (ms): 0.2259



## 7. All three maps

In [9]:
print("MAP_A")
show(MAP_A)

MAP_A
UCS
Path: [(0, 0), (0, 1), (0, 2), (1, 2), (2, 2), (2, 3), (2, 4), (2, 5), (2, 6), (2, 7), (3, 7), (4, 7)]
Moves: 11 | Cost: 15
S**#....
##*#.##.
..******
.####.#*
...~..#G
Nodes expanded: 28 | Max frontier: 4 | Time (ms): 0.1685

A*
Path: [(0, 0), (0, 1), (0, 2), (1, 2), (2, 2), (2, 3), (2, 4), (2, 5), (2, 6), (2, 7), (3, 7), (4, 7)]
Moves: 11 | Cost: 15
S**#....
##*#.##.
..******
.####.#*
...~..#G
Nodes expanded: 21 | Max frontier: 5 | Time (ms): 0.2533



In [10]:
print("MAP_B")
show(MAP_B)

MAP_B
UCS
Path: [(0, 0), (1, 0), (2, 0), (2, 1), (2, 2), (2, 3), (2, 4), (2, 5), (2, 6), (2, 7), (2, 8), (1, 8), (0, 8)]
Moves: 12 | Cost: 12
S~~~~~~~G
*#######*
*********
.#.#.#.#.
.........
Nodes expanded: 30 | Max frontier: 4 | Time (ms): 0.3228

A*
Path: [(0, 0), (1, 0), (2, 0), (2, 1), (2, 2), (2, 3), (2, 4), (2, 5), (2, 6), (2, 7), (2, 8), (1, 8), (0, 8)]
Moves: 12 | Cost: 12
S~~~~~~~G
*#######*
*********
.#.#.#.#.
.........
Nodes expanded: 15 | Max frontier: 7 | Time (ms): 0.1692



In [11]:
print("MAP_C")
show(MAP_C)

MAP_C
UCS
No path found
Nodes expanded: 13 | Max frontier: 3 | Time (ms): 0.146

A*
No path found
Nodes expanded: 13 | Max frontier: 3 | Time (ms): 0.1587



## 8. Results table

In [12]:
maps = {"MAP_A": MAP_A, "MAP_B": MAP_B, "MAP_C": MAP_C}
algorithms = {"UCS": ucs, "A*": astar}

print("Map | Algo | Moves | Cost | Expanded | Max frontier | Time (ms)")
for map_name, grid in maps.items():
    for algo_name, algo in algorithms.items():
        s = run_and_measure(algo, grid)
        print(map_name, "|", algo_name, "|", s["moves"], "|", s["cost"], "|",
              s["expanded"], "|", s["frontier"], "|", round(s["time_ms"], 4))

Map | Algo | Moves | Cost | Expanded | Max frontier | Time (ms)
MAP_A | UCS | 11 | 15 | 28 | 4 | 0.278
MAP_A | A* | 11 | 15 | 21 | 5 | 0.2571
MAP_B | UCS | 12 | 12 | 30 | 4 | 0.2444
MAP_B | A* | 12 | 12 | 15 | 7 | 0.1749
MAP_C | UCS | None | None | 13 | 3 | 0.3877
MAP_C | A* | None | None | 13 | 3 | 0.1525


### Comparison of UCS and A*

Both Uniform Cost Search (UCS) and A* found paths with the same optimal total cost on MAP_A and MAP_B. On MAP_A, both algorithms found a path with 11 moves and a cost of 15. On MAP_B, both found a path with 12 moves and a cost of 12, which is cheaper than the direct route through rough terrain, despite requiring more moves. Neither algorithm found a path on MAP_C, as the goal is unreachable.


A* expanded fewer nodes than UCS on both solvable maps: 21 compared with 28 on MAP_A, and 15 compared with 30 on MAP_B. This demonstrates how the Manhattan-distance heuristic helps A* focus its search toward the goal. For this warehouse robot, I would choose A* because it found equally cheap paths while expanding fewer nodes in these tests. UCS remains a useful alternative when a heuristic is unavailable.